In [2]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load all saved models
best_model = joblib.load('best_model.pkl')
scaler_clf = joblib.load('scaler_clf.pkl')
le         = joblib.load('label_encoder.pkl')
df         = pd.read_csv('df_clustered.csv')

print(" Models loaded")
print(f"  Classes : {le.classes_}")


 Models loaded
  Classes : ['Overpriced Low Performer' 'Smart Discounter' 'Underperformer']


#  Strategy Profiles (from clustering)

In [3]:
# Build strategy profiles from real data
# These will power the recommendations in the report

X_features = [
    'price', 'shippingCost', 'rating',
    'category_id', 'price_rank_in_category',
    'price_vs_category_avg', 'effective_price'
]

# Strategy stats from real data
strategy_profiles = df.groupby('strategy').agg(
    avg_sold          = ('sold',              'mean'),
    avg_discount      = ('discount_aggressiveness', 'mean'),
    avg_price         = ('price',             'mean'),
    avg_rating        = ('rating',            'mean'),
    avg_shipping      = ('shippingCost',      'mean'),
    product_count     = ('price',             'count')
).round(2)

print("Strategy Profiles from Real Data:")
print(strategy_profiles.to_string())

# Save for app
strategy_profiles.to_csv('strategy_profiles.csv')
print("\n Saved strategy_profiles.csv")

Strategy Profiles from Real Data:
                          avg_sold  avg_discount  avg_price  avg_rating  avg_shipping  product_count
strategy                                                                                            
Overpriced Low Performer      7.27          1.70    1251.92        1.32        207.01          58048
Smart Discounter            131.78          6.63      46.05        4.77         13.55         261124
Underperformer                2.12          1.81      94.46        0.70         22.45         391996

 Saved strategy_profiles.csv


#  Best Strategy Per Category 

In [4]:
# For each category = best strategy + avg sold
print("start")
best_per_category = (
    df.groupby(['category_name', 'strategy'])['sold']
    .mean()
    .reset_index()
    .sort_values(['category_name', 'sold'], ascending=[True, False])
    .groupby('category_name')
    .first()
    .reset_index()
    .rename(columns={'strategy': 'best_strategy', 'sold': 'avg_sold'})
)

print(f" Best strategy computed for {len(best_per_category)} categories")
print(best_per_category.head(10).to_string(index=False))

# Save for app
best_per_category.to_csv('best_per_category.csv', index=False)
print("\n Saved best_per_category.csv")

start
 Best strategy computed for 325 categories
                     category_name    best_strategy   avg_sold
360-deg--video-cameras-accessories Smart Discounter 111.580153
                    access-control Smart Discounter  66.358949
                 accessories-parts Smart Discounter 155.037525
                action-toy-figures Smart Discounter 114.120515
                 active-components Smart Discounter  85.680645
                     activity-gear Smart Discounter  83.420152
                      art-supplies Smart Discounter  87.957323
              arts-crafts-diy-toys Smart Discounter 122.511864
                arts-crafts-sewing Smart Discounter 122.083333
         atv-rv-boat-other-vehicle Smart Discounter  37.162409

 Saved best_per_category.csv


 # Discount Recommendations Per Strategy

In [8]:
# What discount range does each strategy use?
# This powers the "recommended discount" in the report

discount_recommendations = df.groupby('strategy')['discount_aggressiveness'].agg(
    ['mean', 'median', 'std']
).round(2)

# Map strategy to readable discount advice
strategy_advice = {
    'Smart Discounter': {
        'discount_range' : '25% – 50%',
        'shipping'       : 'Free shipping strongly recommended',
        'pricing'        : 'Keep price affordable vs category average',
        'key_action'     : 'Combine discount + free shipping for maximum impact',
        'color'          : '#1A3A5C'
    },
    'Overpriced Low Performer': {
        'discount_range' : '0% – 15%',
        'shipping'       : 'Shipping cost less critical',
        'pricing'        : 'Premium pricing — justify with quality',
        'key_action'     : 'Improve rating before increasing price',
        'color'          : '#378ADD'
    },
    'Underperformer': {
        'discount_range' : '10% – 25%',
        'shipping'       : 'Reduce shipping cost',
        'pricing'        : 'Reposition price closer to category average',
        'key_action'     : 'Focus on improving product rating first',
        'color'          : '#B4B2A9'
    }
}

print("Strategy Advice:")
for strategy, advice in strategy_advice.items():
    print(f"\n{strategy}:")
    for key, val in advice.items():
        if key != 'color':
            print(f"  {key:<15} : {val}")

Strategy Advice:

Smart Discounter:
  discount_range  : 25% – 50%
  shipping        : Free shipping strongly recommended
  pricing         : Keep price affordable vs category average
  key_action      : Combine discount + free shipping for maximum impact

Overpriced Low Performer:
  discount_range  : 0% – 15%
  shipping        : Shipping cost less critical
  pricing         : Premium pricing — justify with quality
  key_action      : Improve rating before increasing price

Underperformer:
  discount_range  : 10% – 25%
  shipping        : Reduce shipping cost
  pricing         : Reposition price closer to category average
  key_action      : Focus on improving product rating first


#  Predict Function

In [9]:
# Core prediction function
# Input  : category_name, price, rating, shippingCost
# Output : full pricing report

def predict_strategy(category_name, price, rating, shipping_cost):
    """
    Predicts the best pricing strategy for a product.
    
    Parameters:
        category_name : str   — product category
        price         : float — product price in DT
        rating        : float — product rating (0-5)
        shipping_cost : float — shipping cost (0 = free)
    
    Returns:
        dict — full pricing strategy report
    """
    
    # Get category stats
    cat_data = df[df['category_name'] == category_name]
    
    if len(cat_data) == 0:
        # Unknown category → use global averages
        cat_avg_price = df['price'].mean()
        cat_id        = 0
    else:
        cat_avg_price = cat_data['price'].mean()
        cat_id        = cat_data['category_id'].iloc[0]

    # Engineer features (same as training)
    effective_price       = price * (1 - 0.30)  # assume 30% discount
    price_vs_category_avg = price / cat_avg_price if cat_avg_price > 0 else 1
    price_rank            = (
        cat_data['price'] < price
    ).mean() if len(cat_data) > 0 else 0.5

    # Build feature vector
    features = pd.DataFrame([{
        'price'                 : price,
        'shippingCost'          : shipping_cost,
        'rating'                : rating,
        'category_id'           : cat_id,
        'price_rank_in_category': price_rank,
        'price_vs_category_avg' : price_vs_category_avg,
        'effective_price'       : effective_price
    }])

    # Predict
    prediction     = best_model.predict(features)[0]
    probabilities  = best_model.predict_proba(features)[0]
    strategy_name  = le.inverse_transform([prediction])[0]
    confidence     = probabilities.max() * 100

    # Get advice
    advice = strategy_advice[strategy_name]

    # Get category best strategy
    cat_best = best_per_category[
        best_per_category['category_name'] == category_name
    ]
    category_winner = cat_best['best_strategy'].values[0] \
        if len(cat_best) > 0 else 'Smart Discounter'
    category_avg_sold = cat_best['avg_sold'].values[0] \
        if len(cat_best) > 0 else 0

    # Build report
    report = {
        'strategy'          : strategy_name,
        'confidence'        : f"{confidence:.1f}%",
        'discount_range'    : advice['discount_range'],
        'shipping'          : advice['shipping'],
        'pricing_tip'       : advice['pricing'],
        'key_action'        : advice['key_action'],
        'category_winner'   : category_winner,
        'category_avg_sold' : f"{category_avg_sold:.0f} units",
        'probabilities'     : {
            cls: f"{prob*100:.1f}%"
            for cls, prob in zip(le.classes_, probabilities)
        }
    }

    return report

print(" predict_strategy() is ready")

 predict_strategy() is ready


# Test the Function

In [10]:
# Test with a few examples

test_cases = [
    {
        'category_name': 'consumer-electronics',
        'price'        : 250,
        'rating'       : 4.5,
        'shipping_cost': 0
    },
    {
        'category_name': 'beauty-health',
        'price'        : 50,
        'rating'       : 3.0,
        'shipping_cost': 15
    },
    {
        'category_name': 'laptops',
        'price'        : 1500,
        'rating'       : 2.0,
        'shipping_cost': 50
    }
]

for test in test_cases:
    print("\n" + "=" * 55)
    print(f"INPUT:")
    print(f"  Category     : {test['category_name']}")
    print(f"  Price        : {test['price']} DT")
    print(f"  Rating       : {test['rating']}")
    print(f"  Shipping     : {test['shipping_cost']} DT")
    print(f"\nOUTPUT:")

    report = predict_strategy(**test)

    print(f"  Strategy     : {report['strategy']}")
    print(f"  Confidence   : {report['confidence']}")
    print(f"  Discount     : {report['discount_range']}")
    print(f"  Shipping tip : {report['shipping']}")
    print(f"  Key action   : {report['key_action']}")
    print(f"  Cat winner   : {report['category_winner']}")
    print(f"  Cat avg sold : {report['category_avg_sold']}")
    print(f"  Probabilities: {report['probabilities']}")


INPUT:
  Category     : consumer-electronics
  Price        : 250 DT
  Rating       : 4.5
  Shipping     : 0 DT

OUTPUT:
  Strategy     : Underperformer
  Confidence   : 53.0%
  Discount     : 10% – 25%
  Shipping tip : Reduce shipping cost
  Key action   : Focus on improving product rating first
  Cat winner   : Smart Discounter
  Cat avg sold : 173 units
  Probabilities: {'Overpriced Low Performer': '15.7%', 'Smart Discounter': '31.3%', 'Underperformer': '53.0%'}

INPUT:
  Category     : beauty-health
  Price        : 50 DT
  Rating       : 3.0
  Shipping     : 15 DT

OUTPUT:
  Strategy     : Underperformer
  Confidence   : 74.0%
  Discount     : 10% – 25%
  Shipping tip : Reduce shipping cost
  Key action   : Focus on improving product rating first
  Cat winner   : Smart Discounter
  Cat avg sold : 237 units
  Probabilities: {'Overpriced Low Performer': '7.0%', 'Smart Discounter': '19.0%', 'Underperformer': '74.0%'}

INPUT:
  Category     : laptops
  Price        : 1500 DT
  Rating

# Save Predict Function

In [11]:
import pickle

# Save everything needed for Streamlit app
artifacts = {
    'model'              : best_model,
    'label_encoder'      : le,
    'strategy_advice'    : strategy_advice,
    'best_per_category'  : best_per_category,
    'category_list'      : sorted(df['category_name'].unique().tolist()),
    'df_stats'           : df[['category_name', 'category_id',
                                'price', 'sold']].copy()
}

with open('app_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("Saved app_artifacts.pkl")
print(f"  Categories available : {len(artifacts['category_list'])}")
print(f"   Ready ")

Saved app_artifacts.pkl
  Categories available : 325
   Ready 
